In [2]:
import xml.etree.ElementTree as ET
import csv
from collections import defaultdict

XML_FILE = r"D:\Graduation Project\export\apple_health_export\export_cda.xml"
OUTPUT_CSV = r"D:\Graduation Project\export\cda_observation_catalog.csv"

observation_counter = defaultdict(int)

context = ET.iterparse(XML_FILE, events=("start", "end"))

# Grab namespace from the root
event, root = next(context)
if root.tag.startswith("{"):
    ns = root.tag.split("}")[0] + "}"
else:
    ns = ""

for event, elem in context:
    if event == "end" and elem.tag == f"{ns}observation":

        code = elem.find(f".//{ns}code")
        if code is not None:
            display_name = code.attrib.get("displayName", "UNKNOWN")
            code_value = code.attrib.get("code", "UNKNOWN")

            key = f"{display_name} | {code_value}"
            observation_counter[key] += 1

        elem.clear()

# Save results
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Observation", "Count"])
    for obs, count in sorted(observation_counter.items(), key=lambda x: -x[1]):
        writer.writerow([obs, count])

print("CDA observation catalog created successfully.")


CDA observation catalog created successfully.


In [4]:
import xml.etree.ElementTree as ET
import csv
import os
from datetime import datetime

XML_FILE = r"D:\Graduation Project\export\apple_health_export\export_cda.xml"
OUTPUT_DIR = r"D:\Graduation Project\export\cda_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map CDA display names to output CSV files
OBSERVATION_MAP = {
    "Blood glucose level": "cda_blood_glucose.csv",
    "Heart rate": "cda_heart_rate.csv",
    "Respiratory rate": "cda_respiratory_rate.csv",
    "Oxygen saturation": "cda_oxygen_saturation.csv",
    "Body weight Measured": "cda_weight.csv",
    "Height": "cda_height.csv",
}

# Open CSV writers
writers = {}
files = {}

for csv_name in OBSERVATION_MAP.values():
    path = os.path.join(OUTPUT_DIR, csv_name)
    f = open(path, "w", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=["timestamp", "value", "unit"])
    writer.writeheader()
    writers[csv_name] = writer
    files[csv_name] = f

def parse_cda_time(cda_time):
    # YYYYMMDDHHMMSS+ZZZZ → datetime
    return datetime.strptime(cda_time[:14], "%Y%m%d%H%M%S")

# Namespace-safe streaming parse
context = ET.iterparse(XML_FILE, events=("start", "end"))
event, root = next(context)
ns = root.tag.split("}")[0] + "}"

for event, elem in context:
    if event == "end" and elem.tag == f"{ns}observation":

        code = elem.find(f"{ns}code")
        if code is None:
            elem.clear()
            continue

        display_name = code.attrib.get("displayName")
        if display_name not in OBSERVATION_MAP:
            elem.clear()
            continue

        # Timestamp
        low = elem.find(f".//{ns}effectiveTime/{ns}low")
        if low is None:
            elem.clear()
            continue

        timestamp = parse_cda_time(low.attrib.get("value"))

        # Value
        value_elem = elem.find(f"{ns}value")
        if value_elem is None:
            elem.clear()
            continue

        value = value_elem.attrib.get("value")
        unit = value_elem.attrib.get("unit")

        csv_name = OBSERVATION_MAP[display_name]
        writers[csv_name].writerow({
            "timestamp": timestamp,
            "value": value,
            "unit": unit
        })

        elem.clear()

# Close files
for f in files.values():
    f.close()

print("CDA parsing completed successfully.")


CDA parsing completed successfully.


In [5]:
import pandas as pd

HR_FILE = r"D:\Graduation Project\export\cda_outputs\cda_heart_rate.csv"
BG_FILE = r"D:\Graduation Project\export\cda_outputs\cda_blood_glucose.csv"
OUTPUT_FILE = r"D:\Graduation Project\export\cda_hr_bg_merged.csv"

# Load CSVs
hr = pd.read_csv(HR_FILE)
bg = pd.read_csv(BG_FILE)

# Parse timestamps (timezone-safe, normalize)
hr['timestamp'] = pd.to_datetime(hr['timestamp'], utc=True).dt.tz_convert(None)
bg['timestamp'] = pd.to_datetime(bg['timestamp'], utc=True).dt.tz_convert(None)

# Rename value columns
hr = hr.rename(columns={'value': 'heart_rate'})[['timestamp', 'heart_rate']]
bg = bg.rename(columns={'value': 'blood_glucose'})[['timestamp', 'blood_glucose']]

# Merge (outer join)
merged = pd.merge(hr, bg, on='timestamp', how='outer')

# Sort by time
merged = merged.sort_values('timestamp')

# Save
merged.to_csv(OUTPUT_FILE, index=False)

print("Merged CDA heart rate + blood glucose dataset created.")


Merged CDA heart rate + blood glucose dataset created.
